![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 20 -- Lab 1: Audio Classification for Respiratory Disease Detection

**Scenario:** A hospital wants to build an AI system that can detect respiratory diseases (asthma, pneumonia, COPD) just from the sound of a patient breathing or coughing -- recorded on a regular phone. Your job is to build two classifiers and compare them.

You will:
1. Explore audio data visually -- waveforms, spectrograms, mel-spectrograms
2. Build a **1D CNN** from scratch (like Day 9, but for audio instead of images)
3. Fine-tune **HuBERT** (a pre-trained audio model) using **LoRA** (from Day 19)
4. Compare: does a pre-trained model beat your CNN?

| Part | Goal |
|---|---|
| Part 1 | Load and explore the dataset |
| Part 2 | EDA: listen, plot waveforms, build spectrograms |
| Part 3 | Understand Conv1d (1D convolutions for audio) |
| Part 4 | Build and train a 1D CNN classifier |
| Part 5 | Fine-tune HuBERT with LoRA |
| Part 6 | Compare both models |

In [ ]:
!pip uninstall torchao -y -q
!pip install kagglehub torchaudio peft -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchaudio
import torchaudio.transforms as T

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

import os
import glob
import random
import IPython.display as ipd

plt.rcParams['figure.dpi'] = 120

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
# Part 1 -- Load and Explore the Dataset (GIVEN)

We are using the **Asthma Detection Dataset v2** from Kaggle. It contains WAV recordings of breathing sounds from patients with different respiratory conditions, recorded using regular phone microphones.

There are 5 classes:
- **Asthma** -- wheezing, tight breathing
- **Bronchial** -- inflamed airways
- **COPD** -- chronic obstructive pulmonary disease
- **Healthy** -- normal breathing
- **Pneumonia** -- fluid in lungs

In [ ]:
# --- GIVEN: Download the dataset ---
import kagglehub

path = kagglehub.dataset_download("mohammedtawfikmusaed/asthma-detection-dataset-version-2")
print(f"Dataset downloaded to: {path}")
print()

# Explore the folder structure
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = '  ' * level
    folder_name = os.path.basename(root)
    audio_count = len([f for f in files if f.lower().endswith(('.wav', '.mp3', '.flac', '.ogg'))])
    if audio_count > 0:
        print(f"{indent}{folder_name}/ ({audio_count} audio files)")
    elif dirs:
        print(f"{indent}{folder_name}/")

# Auto-detect the data root (kagglehub sometimes nests files in subfolders)
data_root = path
for root, dirs, files in os.walk(path):
    audio_dirs = []
    for d in dirs:
        subdir = os.path.join(root, d)
        has_audio = any(
            f.lower().endswith(('.wav', '.mp3', '.flac', '.ogg'))
            for f in os.listdir(subdir) if os.path.isfile(os.path.join(subdir, f))
        )
        if has_audio:
            audio_dirs.append(d)
    if len(audio_dirs) >= 2:
        data_root = root
        break

print(f"\nData root: {data_root}")

In [ ]:
# --- GIVEN: Find all audio files and their labels ---
CLASS_NAMES = sorted([d for d in os.listdir(data_root)
                      if os.path.isdir(os.path.join(data_root, d)) and not d.startswith('.')])
print(f"Classes found: {CLASS_NAMES}")

audio_files = []
labels = []

AUDIO_EXTENSIONS = ('.wav', '.mp3', '.flac', '.ogg')

for class_idx, class_name in enumerate(CLASS_NAMES):
    class_dir = os.path.join(data_root, class_name)
    class_files = []
    for ext in AUDIO_EXTENSIONS:
        class_files.extend(glob.glob(os.path.join(class_dir, '**', f'*{ext}'), recursive=True))
    audio_files.extend(class_files)
    labels.extend([class_idx] * len(class_files))
    print(f"  {class_name}: {len(class_files)} files")

print(f"\nTotal: {len(audio_files)} audio files")

In [ ]:
# --- GIVEN: Class distribution bar chart ---
class_counts = [labels.count(i) for i in range(len(CLASS_NAMES))]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CLASS_NAMES, class_counts, color=['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12'])
ax.set_ylabel('Number of Samples')
ax.set_title('Class Distribution')
for bar, count in zip(bars, class_counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            str(count), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print("Note: The dataset is imbalanced -- COPD has the most samples, Bronchial the fewest.")
print("We will use class weights during training to handle this.")

In [ ]:
# --- GIVEN: Load one sample and check its properties ---
sample_waveform, sample_rate = torchaudio.load(audio_files[0])
print(f"Sample file: {os.path.basename(audio_files[0])}")
print(f"Label: {CLASS_NAMES[labels[0]]}")
print(f"Waveform shape: {sample_waveform.shape}  (channels x samples)")
print(f"Sample rate: {sample_rate} Hz")
print(f"Duration: {sample_waveform.shape[1] / sample_rate:.2f} seconds")
print(f"Value range: [{sample_waveform.min():.4f}, {sample_waveform.max():.4f}]")

---
# Part 2 -- Exploratory Data Analysis (EDA)

Before building any model, we need to **look at** and **listen to** our data. This is especially important for audio -- you can often hear differences between classes that will help you understand what the model needs to learn.

## Task 1: Listen to Audio Samples

**TODO:** For each of the 5 classes, pick one audio file and play it using `IPython.display.Audio`.

Hint: `ipd.Audio(waveform.numpy(), rate=sample_rate)` or `ipd.Audio(filepath)`

Listen carefully -- can you hear any differences between healthy and asthma breathing?

In [ ]:
# Your code here


## Task 2: Plot Waveforms

**TODO:** Create a figure with 5 subplots (one per class). For each class, load one audio file and plot its waveform (amplitude vs time).

Hint:
- Load audio: `waveform, sr = torchaudio.load(filepath)`
- Time axis: `time = torch.arange(waveform.shape[1]) / sr`
- Plot: `ax.plot(time, waveform[0])`
- Set each subplot title to the class name

In [ ]:
# Your code here


## Task 3: Build and Plot Spectrograms

**TODO:** Pick one Healthy sample and one Asthma sample. For each, create and plot:
1. A **spectrogram** (frequency vs time)
2. A **mel-spectrogram** (mel frequency vs time)

This should give you a 2x2 grid: (Healthy spectrogram, Asthma spectrogram) on top, (Healthy mel-spectrogram, Asthma mel-spectrogram) on bottom.

Hints:
```python
# Spectrogram
spectrogram_transform = T.Spectrogram(n_fft=1024, hop_length=512)
spec = spectrogram_transform(waveform)
spec_db = torchaudio.functional.amplitude_to_DB(spec, multiplier=10, amin=1e-10, db_multiplier=0)

# Mel-spectrogram
mel_transform = T.MelSpectrogram(sample_rate=sr, n_fft=1024, hop_length=512, n_mels=80)
mel_spec = mel_transform(waveform)
mel_spec_db = torchaudio.functional.amplitude_to_DB(mel_spec, multiplier=10, amin=1e-10, db_multiplier=0)

# Plot with: ax.imshow(spec_db[0], aspect='auto', origin='lower', cmap='viridis')
```

In [ ]:
# Your code here


---
# Part 3 -- From Conv2d to Conv1d (GIVEN)

On Day 9, you built a CNN for images using `Conv2d` -- filters that slide over a **2D grid** (height x width).

Audio is different: it is a **1D signal** (just time). So we use `Conv1d`, which slides a filter along **one axis**.

| | Images (Day 9) | Audio (Today) |
|---|---|---|
| Input shape | (channels, height, width) | (channels, time) |
| Convolution | Conv2d -- slides in 2 directions | Conv1d -- slides along time |
| Pooling | MaxPool2d | MaxPool1d |
| BatchNorm | BatchNorm2d | BatchNorm1d |
| Example input | (1, 28, 28) for MNIST | (1, 64000) for 4s audio at 16kHz |

In [ ]:
# --- GIVEN: Demo -- Conv1d on a waveform ---
# Load a sample and resample to 16kHz
demo_waveform, demo_sr = torchaudio.load(audio_files[0])
if demo_sr != 16000:
    resampler = T.Resample(demo_sr, 16000)
    demo_waveform = resampler(demo_waveform)
    demo_sr = 16000

# Take first channel, trim to 4 seconds
TARGET_LENGTH = 4 * 16000  # 4 seconds at 16kHz = 64,000 samples
demo_waveform = demo_waveform[0:1, :TARGET_LENGTH]
if demo_waveform.shape[1] < TARGET_LENGTH:
    demo_waveform = F.pad(demo_waveform, (0, TARGET_LENGTH - demo_waveform.shape[1]))

print(f"Input shape: {demo_waveform.shape}  (1 channel, {TARGET_LENGTH} time steps)")

# Apply one Conv1d layer
conv = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=80, stride=4)
with torch.no_grad():
    output = conv(demo_waveform.unsqueeze(0))  # add batch dim

print(f"After Conv1d(1->16, kernel=80, stride=4): {output.shape}")
print(f"  16 filters, each producing a 1D feature map of length {output.shape[2]}")
print(f"  (64000 - 80) / 4 + 1 = {(64000 - 80) // 4 + 1} time steps")

In [ ]:
# --- GIVEN: Visualize what Conv1d does ---
fig, axes = plt.subplots(2, 1, figsize=(12, 5))

# Original waveform
axes[0].plot(demo_waveform[0].numpy(), linewidth=0.5)
axes[0].set_title('Original Waveform (1 channel, 64000 samples)')
axes[0].set_xlabel('Sample')

# Conv1d output (show first 4 filters as colored lines)
out_np = output[0].detach().numpy()
for i in range(4):
    axes[1].plot(out_np[i], linewidth=0.5, alpha=0.7, label=f'Filter {i}')
axes[1].set_title(f'After Conv1d: 16 feature maps, {out_np.shape[1]} time steps (showing 4)')
axes[1].set_xlabel('Time step')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Each Conv1d filter detects a different pattern in the audio.")
print("Some filters might respond to high frequencies, others to low.")
print("This is exactly the same idea as Conv2d detecting edges and textures in images!")

---
# Part 4 -- Build and Train a 1D CNN Classifier

Now we build a full 1D CNN to classify respiratory audio. The architecture follows the same pattern as Day 9 (conv blocks + classifier), just with Conv1d instead of Conv2d.

## Dataset Preparation (GIVEN)

We need to:
1. Load each WAV file
2. Resample to 16kHz (standard for speech/audio AI)
3. Pad or trim to exactly 4 seconds (64,000 samples)
4. Split into train and test sets

In [ ]:
# --- GIVEN: Audio Dataset class ---
class RespiratoryDataset(Dataset):
    def __init__(self, file_paths, labels, target_sr=16000, duration_sec=4):
        self.file_paths = file_paths
        self.labels = labels
        self.target_sr = target_sr
        self.target_length = target_sr * duration_sec

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        waveform, sr = torchaudio.load(self.file_paths[idx])

        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample if needed
        if sr != self.target_sr:
            resampler = T.Resample(sr, self.target_sr)
            waveform = resampler(waveform)

        # Pad or trim to fixed length
        if waveform.shape[1] > self.target_length:
            waveform = waveform[:, :self.target_length]
        elif waveform.shape[1] < self.target_length:
            waveform = F.pad(waveform, (0, self.target_length - waveform.shape[1]))

        return waveform, self.labels[idx]


# Train/test split (stratified to keep class ratios)
train_files, test_files, train_labels, test_labels = train_test_split(
    audio_files, labels, test_size=0.2, random_state=SEED, stratify=labels
)

train_dataset = RespiratoryDataset(train_files, train_labels)
test_dataset = RespiratoryDataset(test_files, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} samples")
print(f"Test:  {len(test_dataset)} samples")

# Check one batch
sample_batch, sample_labels = next(iter(train_loader))
print(f"\nBatch shape: {sample_batch.shape}  (batch, channels, time)")
print(f"Labels: {sample_labels}")

In [ ]:
# --- GIVEN: Compute class weights for imbalanced data ---
from collections import Counter

label_counts = Counter(train_labels)
total = len(train_labels)
class_weights = torch.tensor(
    [total / (len(CLASS_NAMES) * label_counts[i]) for i in range(len(CLASS_NAMES))],
    dtype=torch.float32
).to(device)

print("Class weights (higher = rarer class gets more importance):")
for name, weight in zip(CLASS_NAMES, class_weights):
    print(f"  {name}: {weight:.2f}")

## Task 4: Build the 1D CNN

**TODO:** Fill in the `None` values to complete the CNN architecture.

The architecture has 3 conv blocks followed by a classifier, exactly like Day 9 but with Conv1d:

- **Block 1:** Conv1d(1 -> 32, kernel=80, stride=4) -> BatchNorm1d -> ReLU -> MaxPool1d(4)
- **Block 2:** Conv1d(32 -> 64, kernel=3, padding=1) -> BatchNorm1d -> ReLU -> MaxPool1d(4)
- **Block 3:** Conv1d(64 -> 128, kernel=3, padding=1) -> BatchNorm1d -> ReLU -> MaxPool1d(4)
- **Classifier:** AdaptiveAvgPool1d(1) -> Flatten -> Linear(128, 64) -> ReLU -> Dropout(0.3) -> Linear(64, 5)

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: input (1, 64000) -> output (32, 4000)
            nn.Conv1d(None, None, kernel_size=80, stride=4),  # TODO: in_channels, out_channels
            nn.BatchNorm1d(None),                              # TODO: num_features
            nn.ReLU(),
            nn.MaxPool1d(None),                                # TODO: pool size

            # Block 2: (32, 4000) -> (64, 1000)
            nn.Conv1d(None, None, kernel_size=3, padding=1),   # TODO: in_channels, out_channels
            nn.BatchNorm1d(None),                              # TODO: num_features
            nn.ReLU(),
            nn.MaxPool1d(None),                                # TODO: pool size

            # Block 3: (64, 1000) -> (128, 250)
            nn.Conv1d(None, None, kernel_size=3, padding=1),   # TODO: in_channels, out_channels
            nn.BatchNorm1d(None),                              # TODO: num_features
            nn.ReLU(),
            nn.MaxPool1d(None),                                # TODO: pool size
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(None, 64),    # TODO: input features
            nn.ReLU(),
            nn.Dropout(None),       # TODO: dropout probability
            nn.Linear(64, None),    # TODO: num_classes
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
# --- GIVEN: Instantiate and verify ---
cnn_model = AudioCNN(num_classes=len(CLASS_NAMES)).to(device)
print(cnn_model)

total_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

# Verify forward pass
dummy_input = torch.randn(4, 1, 64000).to(device)
dummy_output = cnn_model(dummy_input)
print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {dummy_output.shape}  (should be [4, {len(CLASS_NAMES)}])")

## Task 5: Train the CNN

**TODO:** Fill in the `None` values in the training loop. This is the same pattern from Day 9:
- Forward pass
- Compute loss
- Backward pass
- Optimizer step

In [ ]:
NUM_EPOCHS = 20

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(cnn_model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

cnn_train_losses = []
cnn_val_accuracies = []

for epoch in range(NUM_EPOCHS):
    # --- Training ---
    cnn_model.train()
    running_loss = 0.0
    num_batches = 0

    for waveforms, targets in train_loader:
        waveforms, targets = waveforms.to(device), targets.to(device)

        outputs = cnn_model(None)         # TODO: forward pass
        loss = criterion(None, None)       # TODO: compute loss

        optimizer.zero_grad()
        None                               # TODO: backward pass
        None                               # TODO: optimizer step

        running_loss += loss.item()
        num_batches += 1

    avg_loss = running_loss / num_batches
    cnn_train_losses.append(avg_loss)

    # --- Validation ---
    cnn_model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for waveforms, targets in test_loader:
            waveforms, targets = waveforms.to(device), targets.to(device)
            outputs = cnn_model(waveforms)
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

    val_acc = 100.0 * correct / total
    cnn_val_accuracies.append(val_acc)

    scheduler.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1:2d}/{NUM_EPOCHS}]  '
              f'Loss: {avg_loss:.4f}  '
              f'Val Acc: {val_acc:.1f}%')

print(f'\nFinal CNN accuracy: {cnn_val_accuracies[-1]:.1f}%')

In [ ]:
# --- GIVEN: Plot training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(cnn_train_losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('CNN Training Loss')

ax2.plot(cnn_val_accuracies)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('CNN Validation Accuracy')

plt.tight_layout()
plt.show()

---
# Part 5 -- Fine-Tune HuBERT with LoRA

Your CNN was trained from scratch on ~970 samples. But **HuBERT** was pre-trained on **960 hours** of speech audio -- it already understands sound patterns, frequencies, and temporal structures.

Instead of training HuBERT's 95 million parameters from scratch, we use **LoRA** (from Day 19): freeze the pre-trained weights and only train tiny adapter matrices. This:
- Trains much faster (fewer parameters to update)
- Needs less data (the model already knows audio)
- Often works better than training from scratch

In [ ]:
# --- GIVEN: Load HuBERT for classification ---
from transformers import AutoFeatureExtractor, HubertForSequenceClassification

MODEL_NAME = "facebook/hubert-base-ls960"

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
hubert_model = HubertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CLASS_NAMES),
    problem_type="single_label_classification",
)

print(f"HuBERT loaded: {sum(p.numel() for p in hubert_model.parameters()):,} parameters")
print(f"Output classes: {len(CLASS_NAMES)}")

## Task 6: Apply LoRA

**TODO:** Create a LoRA config and apply it to HuBERT. Fill in the `None` values.

Remember from Day 19:
- `r` = rank of the low-rank matrices (smaller = fewer parameters, try 8)
- `lora_alpha` = scaling factor (typically 2x the rank, so 16)
- `lora_dropout` = dropout on LoRA layers (try 0.1)
- `target_modules` = which layers to add LoRA to (use `["q_proj", "v_proj"]` for the attention layers)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=None,                    # TODO: rank
    lora_alpha=None,           # TODO: scaling factor
    lora_dropout=None,         # TODO: dropout
    target_modules=None,       # TODO: which layers to adapt
)

hubert_model = get_peft_model(hubert_model, lora_config)
hubert_model.print_trainable_parameters()
hubert_model = hubert_model.to(device)

In [ ]:
# --- GIVEN: HuBERT Dataset (uses feature extractor instead of raw waveform) ---
class HuBERTDataset(Dataset):
    def __init__(self, file_paths, labels, feature_extractor, target_sr=16000, duration_sec=4):
        self.file_paths = file_paths
        self.labels = labels
        self.feature_extractor = feature_extractor
        self.target_sr = target_sr
        self.target_length = target_sr * duration_sec

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        waveform, sr = torchaudio.load(self.file_paths[idx])

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != self.target_sr:
            resampler = T.Resample(sr, self.target_sr)
            waveform = resampler(waveform)

        if waveform.shape[1] > self.target_length:
            waveform = waveform[:, :self.target_length]
        elif waveform.shape[1] < self.target_length:
            waveform = F.pad(waveform, (0, self.target_length - waveform.shape[1]))

        inputs = self.feature_extractor(
            waveform.squeeze(0).numpy(),
            sampling_rate=self.target_sr,
            return_tensors="pt",
            padding=False,
        )
        return inputs["input_values"].squeeze(0), self.labels[idx]


hubert_train_dataset = HuBERTDataset(train_files, train_labels, feature_extractor)
hubert_test_dataset = HuBERTDataset(test_files, test_labels, feature_extractor)

hubert_train_loader = DataLoader(hubert_train_dataset, batch_size=8, shuffle=True, num_workers=2)
hubert_test_loader = DataLoader(hubert_test_dataset, batch_size=8, shuffle=False, num_workers=2)

print(f"HuBERT train batches: {len(hubert_train_loader)}")
print(f"HuBERT test batches:  {len(hubert_test_loader)}")

## Task 7: Train HuBERT + LoRA

**TODO:** Fill in the `None` values to complete the fine-tuning loop. The structure is the same as the CNN training loop -- only the model and data loader are different.

Note: HuBERT returns a `SequenceClassifierOutput` object. The logits are in `output.logits` and the loss can be computed from those.

In [ ]:
HUBERT_EPOCHS = 10

hubert_optimizer = optim.AdamW(hubert_model.parameters(), lr=2e-4, weight_decay=0.01)
hubert_scheduler = optim.lr_scheduler.CosineAnnealingLR(hubert_optimizer, T_max=HUBERT_EPOCHS)
hubert_criterion = nn.CrossEntropyLoss(weight=class_weights)

hubert_train_losses = []
hubert_val_accuracies = []

for epoch in range(HUBERT_EPOCHS):
    # --- Training ---
    hubert_model.train()
    running_loss = 0.0
    num_batches = 0

    for input_values, targets in hubert_train_loader:
        input_values = input_values.to(device)
        targets = targets.to(device)

        output = hubert_model(None)                    # TODO: pass input_values
        loss = hubert_criterion(None, None)             # TODO: output.logits, targets

        hubert_optimizer.zero_grad()
        None                                            # TODO: backward
        None                                            # TODO: step

        running_loss += loss.item()
        num_batches += 1

    avg_loss = running_loss / num_batches
    hubert_train_losses.append(avg_loss)

    # --- Validation ---
    hubert_model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for input_values, targets in hubert_test_loader:
            input_values = input_values.to(device)
            targets = targets.to(device)
            output = hubert_model(input_values)
            _, predicted = torch.max(output.logits, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

    val_acc = 100.0 * correct / total
    hubert_val_accuracies.append(val_acc)

    hubert_scheduler.step()

    print(f'Epoch [{epoch+1:2d}/{HUBERT_EPOCHS}]  '
          f'Loss: {avg_loss:.4f}  '
          f'Val Acc: {val_acc:.1f}%')

print(f'\nFinal HuBERT+LoRA accuracy: {hubert_val_accuracies[-1]:.1f}%')

In [ ]:
# --- GIVEN: Plot HuBERT training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(hubert_train_losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('HuBERT+LoRA Training Loss')

ax2.plot(hubert_val_accuracies)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('HuBERT+LoRA Validation Accuracy')

plt.tight_layout()
plt.show()

---
# Part 6 -- Compare Both Models

## Task 8: Evaluate and Compare

**TODO:** For each model (CNN and HuBERT+LoRA):
1. Collect all predictions and true labels from the test set
2. Print the classification report (`sklearn.metrics.classification_report`)
3. Plot a confusion matrix (`ConfusionMatrixDisplay`)

Hint for CNN predictions:
```python
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for waveforms, targets in test_loader:
        waveforms = waveforms.to(device)
        outputs = model(waveforms)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(targets.numpy())
```

In [ ]:
# Your code here


In [ ]:
# --- GIVEN: Side-by-side accuracy comparison ---
cnn_final_acc = cnn_val_accuracies[-1]
hubert_final_acc = hubert_val_accuracies[-1]

fig, ax = plt.subplots(figsize=(6, 4))
models = ['1D CNN\n(from scratch)', 'HuBERT + LoRA\n(pre-trained)']
accuracies = [cnn_final_acc, hubert_final_acc]
colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(models, accuracies, color=colors, width=0.5)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('CNN vs HuBERT+LoRA')
ax.set_ylim(0, 100)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{acc:.1f}%', ha='center', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()

print(f"CNN trained from scratch:   {cnn_final_acc:.1f}%")
print(f"HuBERT + LoRA:             {hubert_final_acc:.1f}%")
print(f"\nCNN parameters:    {sum(p.numel() for p in AudioCNN().parameters()):,}")
hubert_trainable = sum(p.numel() for p in hubert_model.parameters() if p.requires_grad)
hubert_total = sum(p.numel() for p in hubert_model.parameters())
print(f"HuBERT total:      {hubert_total:,} (but only {hubert_trainable:,} trained with LoRA)")

---
## Discussion

1. Which model performed better -- the CNN or HuBERT+LoRA? Why do you think that is?
2. HuBERT was pre-trained on **English speech**, but our task is **respiratory sounds**. Why does transfer learning still work here?
3. Look at the confusion matrices. Which disease classes are hardest to tell apart? Does that make medical sense?
4. We used only ~970 training samples. How do you think results would change with 10x more data?
5. Could this system be deployed in a real hospital? What are the risks of getting it wrong?

---
## Wrap-Up

**What you learned:**

| Technique | Where Used | Day Introduced |
|---|---|---|
| Spectrograms and mel-spectrograms | EDA visualization | Day 20 (today) |
| Conv1d (1D convolutions) | AudioCNN model | Day 20 (based on Day 9 Conv2d) |
| MaxPool1d, BatchNorm1d | AudioCNN model | Day 20 (1D versions of Day 9) |
| Class-weighted loss | Imbalanced dataset handling | Day 3 (cross-entropy) |
| AdamW + cosine LR | Optimizer and scheduler | Day 8 |
| Transfer learning | HuBERT pre-trained model | Day 10 |
| LoRA (Low-Rank Adaptation) | Efficient fine-tuning | Day 19 |
| Confusion matrix | Model evaluation | Day 3 |

**Key takeaway:** A pre-trained audio model (HuBERT) fine-tuned with LoRA can outperform a CNN trained from scratch, even on a small dataset of a completely different domain (respiratory sounds vs speech). This is the power of transfer learning -- the same pattern you saw with images on Day 10.